In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemini-3.1-flash-lite",
    model_provider="google-genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
)


In [3]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dataclass
class LanguageContext:
    user_language: str = "English"

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'Dia dhuit! Tá mé go maith, go raibh maith agat. Agus tú féin?', 'extras': {'signature': 'EnEKbwFpFH0Tzya4+z67WnaK0MPGXTH6Vx/f7AirGtSq2AxPvW0Uzpq9cf1OIvMW120tc05d6oMSyomeYBTa1xLYnbmTmAyUCUGbQEA1Ynllocq4PGzn3/NZhwL0nUwc3YhzFVg07uPgZc/LA2XahQ/rpw=='}}]


In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': '¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú, cómo estás? ¿En qué puedo ayudarte hoy?', 'extras': {'signature': 'EnEKbwFpFH0T4xMPSUJLYN4CAZqSm+FS/o+eS+59WkD0RwtORNUea8chqx0E9GG5rodfOxa2nnWT2/1OUwY82oYbZwkPBbbU/IkSejcMJkNntDtVLk1J4CFhAfe1Ls8xMV3Z+2PAOaPueuWnS7JCCHaivg=='}}]


In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="French")
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "Bonjour ! Je vais très bien, merci de demander. Et vous, comment allez-vous aujourd'hui ?", 'extras': {'signature': 'EnEKbwFpFH0T0Q969HfYl5rrZUKPtRiLDGQ1a6bkVNRSRZsbrj6SqCHSjpFgboNVxFHTU1oV3Synct0oh6AoBVr+ZaL9uVGf/Eg2rn1XeBjIPwlMJsyOjWE8XVGj1mgiIq8XDCgBBtoIKZTqaG+Kkh2xQg=='}}]
